# GNR401 — Advanced Hyperspectral Analysis
## RX Anomaly Detection, VCA Endmember Extraction, Spectral Unmixing, MNF Transform, Graph-based Label Propagation, Extended Morphological Attribute Profiles

All algorithms implemented **from scratch** in NumPy — no scikit-learn, no spectral libraries.

In [ ]:
import numpy as np
from scipy.io import loadmat
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import os

os.makedirs('outputs', exist_ok=True)

def load_mat(path):
    data = loadmat(path)
    for k, v in data.items():
        if not k.startswith('__') and isinstance(v, np.ndarray) and v.ndim in (2, 3):
            return v.astype(np.float64)
    return None

def normalize(img):
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-12)

cube = load_mat('Indian_pines_corrected.mat')   # (145, 145, 200)
gt   = load_mat('Indian_pines_gt.mat')           # (145, 145)
H, W, B = cube.shape
N = H * W
X = cube.reshape(N, B)                           # (21025, 200)
print(f'Cube: {cube.shape}  |  GT classes: {np.unique(gt).astype(int)}  |  Pixels: {N}')

---
## 1. Reed–Xiaoli (RX) Anomaly Detector

The global RX detector models the background as a multivariate Gaussian and scores each pixel by its **Mahalanobis distance**:

$$\delta_{RX}(\mathbf{x}) = (\mathbf{x} - \boldsymbol{\mu})^T \, \Sigma^{-1} \, (\mathbf{x} - \boldsymbol{\mu})$$

This requires inverting the full $B \times B$ covariance matrix — numerically stabilised via Tikhonov regularisation.

In [ ]:
def rx_anomaly_detector(X, reg=1e-6):
    """Global RX anomaly detector via regularised Mahalanobis distance."""
    mu = X.mean(axis=0)                         # (B,)
    X_c = X - mu                                 # (N, B)
    cov = (X_c.T @ X_c) / (X_c.shape[0] - 1)   # (B, B)
    # Tikhonov regularisation for numerical stability
    cov += reg * np.eye(cov.shape[0])
    cov_inv = np.linalg.inv(cov)                 # (B, B)
    
    # Mahalanobis distance per pixel: d_i = x_c_i^T @ Sigma^{-1} @ x_c_i
    # Vectorised as row-wise dot of (X_c @ cov_inv) with X_c
    mahal = np.sum((X_c @ cov_inv) * X_c, axis=1)  # (N,)
    return mahal

rx_scores = rx_anomaly_detector(X)
rx_map = rx_scores.reshape(H, W)

# Threshold at 99th percentile
thresh = np.percentile(rx_scores, 99)
anomaly_mask = (rx_map > thresh).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(normalize(np.mean(cube, axis=2)), cmap='gray')
axes[0].set_title('Band-Averaged Image'); axes[0].axis('off')
im = axes[1].imshow(rx_map, cmap='hot')
axes[1].set_title('RX Anomaly Score (Mahalanobis)'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(anomaly_mask, cmap='gray')
axes[2].set_title(f'Anomalies (p > 99th percentile)'); axes[2].axis('off')
plt.suptitle('Reed–Xiaoli Global Anomaly Detector', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/rx_anomaly.png', dpi=150)
plt.show()
print(f'Covariance: {B}x{B} inverted  |  Anomalous pixels: {int(anomaly_mask.sum())} / {N}')

---
## 2. Vertex Component Analysis (VCA) — Endmember Extraction

VCA is a geometric algorithm that finds **pure spectral signatures** (endmembers) by iteratively projecting data onto the **orthogonal complement** of the subspace spanned by already-selected endmembers, then picking the pixel with the maximum projection.

1. Project data onto a lower-dimensional subspace (via SVD).
2. Iteratively: project onto direction orthogonal to current endmembers, select the extremal pixel.
3. The selected pixels approximate the vertices of the spectral simplex.

In [ ]:
def vca(X, p, seed=42):
    """
    Vertex Component Analysis for endmember extraction.
    X: (N, B) spectral data matrix
    p: number of endmembers to extract
    Returns: indices of endmember pixels, endmember spectra (p, B)
    """
    rng = np.random.RandomState(seed)
    N, B = X.shape
    
    # Step 1: Dimensionality reduction via truncated SVD to p dimensions
    X_c = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(X_c, full_matrices=False)
    # Project onto top-p right singular vectors
    R = Vt[:p, :]             # (p, B)
    Y = (R @ X_c.T).T         # (N, p) — data in reduced space
    
    # Step 2: Iterative endmember selection
    indices = []
    A = np.zeros((p, p))       # endmember matrix in reduced space
    
    for i in range(p):
        if i == 0:
            # Random initial direction
            w = rng.randn(p)
        else:
            # Orthogonal projector: project onto null space of current endmembers
            Ai = A[:, :i]                          # (p, i)
            P_orth = np.eye(p) - Ai @ np.linalg.pinv(Ai)  # (p, p)
            w = P_orth @ rng.randn(p)
        
        w = w / (np.linalg.norm(w) + 1e-12)
        
        # Project all pixels onto direction w, pick the extremal one
        projections = Y @ w          # (N,)
        idx = np.argmax(np.abs(projections))
        indices.append(idx)
        A[:, i] = Y[idx]             # store in reduced space
    
    endmembers = X[indices]           # (p, B) in original spectral space
    return np.array(indices), endmembers

n_endmembers = len(np.unique(gt[gt > 0]))
em_indices, endmembers = vca(X, p=n_endmembers)
print(f'Extracted {n_endmembers} endmembers via VCA')

# Visualise endmember locations on the image
em_coords = np.array([(idx // W, idx % W) for idx in em_indices])
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(normalize(np.mean(cube, axis=2)), cmap='gray')
axes[0].scatter(em_coords[:, 1], em_coords[:, 0], c='red', s=40, marker='x', linewidths=1.5)
axes[0].set_title(f'VCA Endmember Pixel Locations ({n_endmembers} extracted)'); axes[0].axis('off')

for i in range(n_endmembers):
    axes[1].plot(endmembers[i], alpha=0.7, linewidth=0.8)
axes[1].set_xlabel('Band Index'); axes[1].set_ylabel('Reflectance')
axes[1].set_title('VCA-Extracted Endmember Spectra')
plt.tight_layout()
plt.savefig('outputs/vca_endmembers.png', dpi=150)
plt.show()

---
## 3. Fully Constrained Linear Spectral Unmixing (FCLS)

Each pixel $\mathbf{x}$ is modelled as a **linear mixture** of endmember spectra $\mathbf{E}$:

$$\mathbf{x} = \mathbf{E} \, \mathbf{a} + \mathbf{n}$$

Subject to:
- **Non-negativity**: $a_i \geq 0$ (physical constraint — no negative abundances)
- **Sum-to-one**: $\sum_i a_i = 1$ (full pixel coverage)

Solved per-pixel via an **active-set NNLS** method with sum-to-one enforcement through augmented least squares.

In [ ]:
def nnls_activeset(A, b, max_iter=200):
    """
    Non-negative least squares via Lawson–Hanson active set method.
    Solves: min ||Ax - b||^2 s.t. x >= 0
    """
    m, n = A.shape
    x = np.zeros(n)
    passive = set()
    
    AtA = A.T @ A
    Atb = A.T @ b
    
    for _ in range(max_iter):
        gradient = Atb - AtA @ x   # negative gradient of 0.5*||Ax-b||^2
        
        # Check KKT: all active vars have non-positive gradient
        active = set(range(n)) - passive
        if not active:
            break
        violations = [(j, gradient[j]) for j in active if gradient[j] > 1e-12]
        if not violations:
            break
        
        # Move variable with largest gradient violation to passive set
        j_star = max(violations, key=lambda t: t[1])[0]
        passive.add(j_star)
        
        # Solve unconstrained subproblem on passive set
        while True:
            p_idx = sorted(passive)
            A_sub = A[:, p_idx]
            z = np.linalg.lstsq(A_sub, b, rcond=None)[0]
            
            if np.all(z >= -1e-12):
                x = np.zeros(n)
                x[p_idx] = np.maximum(z, 0)
                break
            
            # Find infeasible component, remove from passive
            neg_mask = z < -1e-12
            neg_indices = np.array(p_idx)[neg_mask]
            alphas = x[neg_indices] / (x[neg_indices] - z[neg_mask] + 1e-15)
            alpha = np.min(alphas)
            x_old = x[p_idx]
            x[p_idx] = x_old + alpha * (z - x_old)
            
            to_remove = [p_idx[k] for k in range(len(p_idx))
                         if x[p_idx[k]] < 1e-12 and p_idx[k] != j_star]
            for r in to_remove:
                passive.discard(r)
                x[r] = 0.0
            if not to_remove:
                break
    
    return x

def fcls_unmix(pixel, E, delta=50.0):
    """
    Fully Constrained Linear Spectral Unmixing.
    Enforces sum-to-one via augmented row: appends delta*ones to E and delta to pixel.
    Then solves NNLS on the augmented system.
    """
    p = E.shape[0]  # number of endmembers
    # Augment for sum-to-one constraint
    E_aug = np.vstack([E.T, delta * np.ones((1, p))])   # (B+1, p)
    b_aug = np.append(pixel, delta)                      # (B+1,)
    abundances = nnls_activeset(E_aug, b_aug)
    # Renormalise to ensure exact sum-to-one
    s = abundances.sum()
    if s > 1e-12:
        abundances /= s
    return abundances

print(f'Running FCLS unmixing on {N} pixels with {n_endmembers} endmembers...')
abundance_maps = np.zeros((N, n_endmembers))
for i in range(N):
    abundance_maps[i] = fcls_unmix(X[i], endmembers)
    if (i + 1) % 5000 == 0:
        print(f'  {i+1}/{N} pixels unmixed')

abundance_cube = abundance_maps.reshape(H, W, n_endmembers)
rmse = np.sqrt(np.mean((X - abundance_maps @ endmembers)**2))
print(f'Unmixing RMSE: {rmse:.4f}')

# Visualise top 6 abundance maps
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(abundance_cube[:, :, i], cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Endmember {i+1} Abundance'); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('FCLS Abundance Maps (sum-to-one + non-negativity)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/fcls_abundance.png', dpi=150)
plt.show()

---
## 4. Minimum Noise Fraction (MNF) Transform

MNF is a **noise-adjusted PCA**: instead of maximising total variance, it maximises the **signal-to-noise ratio**.

1. Estimate the noise covariance $\Sigma_n$ from spatial differences.
2. Whiten the data: $\mathbf{Z} = \Sigma_n^{-1/2} \mathbf{X}$
3. Apply PCA to the whitened data — the resulting components are ordered by **decreasing SNR**, not variance.

In [ ]:
def estimate_noise_covariance(cube):
    """
    Estimate noise covariance from spatial first-order differences.
    Noise ~ (pixel[i,j] - pixel[i,j+1]) / sqrt(2)  (shift-difference method).
    """
    H, W, B = cube.shape
    # Horizontal differences
    diff_h = (cube[:, 1:, :] - cube[:, :-1, :]) / np.sqrt(2)
    # Vertical differences
    diff_v = (cube[1:, :, :] - cube[:-1, :, :]) / np.sqrt(2)
    
    diffs = np.vstack([diff_h.reshape(-1, B), diff_v.reshape(-1, B)])
    noise_cov = (diffs.T @ diffs) / (diffs.shape[0] - 1)
    return noise_cov

def mnf_transform(cube, n_components=10, reg=1e-6):
    """
    Minimum Noise Fraction transform.
    Returns MNF components ordered by decreasing SNR.
    """
    H, W, B = cube.shape
    X = cube.reshape(-1, B)
    mu = X.mean(axis=0)
    X_c = X - mu
    
    # Step 1: Noise covariance estimation
    Sigma_n = estimate_noise_covariance(cube)
    Sigma_n += reg * np.eye(B)
    
    # Step 2: Noise whitening — Sigma_n = L L^T, then Z = L^{-1} X
    L = np.linalg.cholesky(Sigma_n)
    L_inv = np.linalg.inv(L)
    Z = (L_inv @ X_c.T).T          # (N, B) noise-whitened
    
    # Step 3: PCA on whitened data
    cov_z = (Z.T @ Z) / (Z.shape[0] - 1)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_z)
    # Sort descending — highest eigenvalue = highest SNR
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # SNR per component: eigenvalue - 1 (noise has unit variance after whitening)
    snr = eigenvalues - 1.0
    
    W_mnf = eigenvectors[:, :n_components]
    mnf_data = Z @ W_mnf
    
    return mnf_data.reshape(H, W, n_components), eigenvalues, snr

mnf_result, mnf_eigvals, mnf_snr = mnf_transform(cube, n_components=10)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(normalize(mnf_result[:, :, i]), cmap='viridis')
    ax.set_title(f'MNF-{i+1} (SNR={mnf_snr[i]:.1f})', fontsize=10)
    ax.axis('off')
plt.suptitle('Minimum Noise Fraction Components (ordered by SNR)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/mnf_components.png', dpi=150)
plt.show()

# SNR plot
plt.figure(figsize=(10, 4))
plt.bar(range(1, 21), mnf_snr[:20])
plt.axhline(y=1, color='r', linestyle='--', label='SNR = 1 (noise-dominated)')
plt.xlabel('MNF Component'); plt.ylabel('Signal-to-Noise Ratio')
plt.title('MNF Eigenvalue Spectrum — SNR per Component'); plt.legend()
plt.tight_layout()
plt.savefig('outputs/mnf_snr.png', dpi=150)
plt.show()
print(f'High-SNR components (SNR > 1): {np.sum(mnf_snr > 1)}')
print(f'MNF-1 SNR: {mnf_snr[0]:.1f}  |  MNF-10 SNR: {mnf_snr[9]:.1f}')

---
## 5. Graph-Based Semi-Supervised Label Propagation

Constructs a **spectral similarity graph** (k-NN with Gaussian kernel weights), then propagates sparse ground-truth labels to all unlabelled pixels by solving the **normalised graph Laplacian** system:

$$\mathbf{F}^* = (\mathbf{I} - \alpha \, \mathbf{D}^{-1/2} \mathbf{W} \mathbf{D}^{-1/2})^{-1} \mathbf{Y}$$

This is equivalent to label diffusion on the graph with propagation rate $\alpha$.

In [ ]:
def build_knn_graph(X, k=10, sigma_scale=1.0):
    """
    Build a symmetric k-NN graph with Gaussian edge weights.
    Uses chunked distance computation to avoid OOM on large N.
    """
    N = X.shape[0]
    rows, cols, vals = [], [], []
    chunk_size = 1000
    
    # Estimate sigma from mean k-NN distance (adaptive bandwidth)
    sample_idx = np.random.choice(N, min(500, N), replace=False)
    sample_dists = np.sqrt(((X[sample_idx, None, :] - X[None, sample_idx, :]) ** 2).sum(axis=2))
    np.fill_diagonal(sample_dists, np.inf)
    sigma = sigma_scale * np.mean(np.sort(sample_dists, axis=1)[:, :k])
    print(f'  Adaptive sigma: {sigma:.4f}')
    
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        # Squared Euclidean distance: ||x_i - x_j||^2
        chunk = X[start:end]                        # (chunk, B)
        d2 = np.sum(chunk**2, axis=1, keepdims=True) + \
             np.sum(X**2, axis=1, keepdims=True).T - \
             2 * (chunk @ X.T)                       # (chunk, N)
        d2 = np.maximum(d2, 0)
        
        # Zero out self-distances
        for i in range(end - start):
            d2[i, start + i] = np.inf
        
        # k-NN selection
        knn_idx = np.argpartition(d2, k, axis=1)[:, :k]
        for i in range(end - start):
            gi = start + i
            for j_local in range(k):
                gj = knn_idx[i, j_local]
                w = np.exp(-d2[i, gj] / (2 * sigma**2))
                rows.append(gi); cols.append(gj); vals.append(w)
                rows.append(gj); cols.append(gi); vals.append(w)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    return W, sigma

def label_propagation(W, labels_sparse, alpha=0.85, max_iter=30, tol=1e-4):
    """
    Semi-supervised label propagation on a graph.
    labels_sparse: (N,) array with class labels (0 = unlabelled, 1..C = classes)
    Uses iterative approach: F_{t+1} = alpha * S @ F_t + (1-alpha) * Y
    where S = D^{-1/2} W D^{-1/2} is the normalised adjacency.
    """
    N = W.shape[0]
    classes = np.unique(labels_sparse[labels_sparse > 0]).astype(int)
    C = len(classes)
    
    # One-hot encode labelled pixels
    Y = np.zeros((N, C))
    for idx, c in enumerate(classes):
        Y[labels_sparse == c, idx] = 1.0
    
    # Normalised graph: S = D^{-1/2} W D^{-1/2}
    d = np.array(W.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0)
    D_inv_sqrt = diags(d_inv_sqrt)
    S = D_inv_sqrt @ W @ D_inv_sqrt
    
    # Iterative propagation
    F = Y.copy()
    for t in range(max_iter):
        F_new = alpha * (S @ F) + (1 - alpha) * Y
        delta = np.max(np.abs(F_new - F))
        F = F_new
        if delta < tol:
            print(f'  Converged at iteration {t+1} (delta={delta:.6f})')
            break
    
    predicted = np.zeros(N, dtype=int)
    labelled_mask = F.max(axis=1) > 1e-8
    predicted[labelled_mask] = classes[np.argmax(F[labelled_mask], axis=1)]
    return predicted, F

# Subsample labels: keep only 5% of GT as training labels
gt_flat = gt.flatten().astype(int)
rng = np.random.RandomState(42)
sparse_labels = np.zeros(N, dtype=int)
for c in np.unique(gt_flat[gt_flat > 0]):
    c_pixels = np.where(gt_flat == c)[0]
    n_keep = max(1, int(0.05 * len(c_pixels)))
    chosen = rng.choice(c_pixels, n_keep, replace=False)
    sparse_labels[chosen] = c

pct_labelled = 100 * np.sum(sparse_labels > 0) / np.sum(gt_flat > 0)
print(f'Using {pct_labelled:.1f}% of GT labels ({np.sum(sparse_labels > 0)} / {np.sum(gt_flat > 0)})')

# Use MNF features for the graph (better SNR than raw bands)
X_mnf = mnf_result.reshape(N, -1)
X_mnf_norm = (X_mnf - X_mnf.mean(axis=0)) / (X_mnf.std(axis=0) + 1e-8)

print('Building k-NN graph on MNF features...')
W_graph, sigma = build_knn_graph(X_mnf_norm, k=15)
print(f'Graph: {W_graph.shape[0]} nodes, {W_graph.nnz} edges')

print('Running label propagation...')
pred_labels, F_scores = label_propagation(W_graph, sparse_labels, alpha=0.90)

# Accuracy on pixels with GT > 0
valid = gt_flat > 0
acc = np.mean(pred_labels[valid] == gt_flat[valid]) * 100
print(f'Label propagation accuracy: {acc:.2f}% (from {pct_labelled:.1f}% labels)')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(gt, cmap='nipy_spectral')
axes[0].set_title('Full Ground Truth'); axes[0].axis('off')
axes[1].imshow(sparse_labels.reshape(H, W), cmap='nipy_spectral')
axes[1].set_title(f'Sparse Labels ({pct_labelled:.1f}% of GT)'); axes[1].axis('off')
axes[2].imshow(pred_labels.reshape(H, W), cmap='nipy_spectral')
axes[2].set_title(f'Propagated Labels ({acc:.1f}% acc)'); axes[2].axis('off')
plt.suptitle('Graph-Based Semi-Supervised Label Propagation (MNF + k-NN)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/label_propagation.png', dpi=150)
plt.show()

---
## 6. Extended Morphological Attribute Profiles (EMAPs)

Spectral-spatial feature extraction:
1. Reduce bands via PCA to top components.
2. On each component, compute **morphological attribute profiles** — apply area-based attribute filters at multiple thresholds (both opening and closing).
3. Stack all profiles into a high-dimensional **spectral-spatial feature vector** per pixel.
4. Classify using a simple nearest-centroid classifier on the EMAP features.

In [ ]:
def connected_components_labeling(binary):
    """Two-pass connected components labeling (4-connectivity) from scratch."""
    H, W = binary.shape
    labels = np.zeros((H, W), dtype=int)
    parent = {}
    
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]  # path compression
            x = parent[x]
        return x
    
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    
    current_label = 0
    
    # First pass
    for i in range(H):
        for j in range(W):
            if not binary[i, j]:
                continue
            neighbors = []
            if i > 0 and labels[i-1, j] > 0:
                neighbors.append(labels[i-1, j])
            if j > 0 and labels[i, j-1] > 0:
                neighbors.append(labels[i, j-1])
            
            if not neighbors:
                current_label += 1
                labels[i, j] = current_label
                parent[current_label] = current_label
            else:
                min_label = min(neighbors)
                labels[i, j] = min_label
                for n in neighbors:
                    union(n, min_label)
    
    # Second pass — resolve equivalences
    for i in range(H):
        for j in range(W):
            if labels[i, j] > 0:
                labels[i, j] = find(labels[i, j])
    
    return labels

def area_attribute_filter(image, threshold, mode='opening'):
    """
    Area-based morphological attribute filter.
    - Opening: removes bright regions smaller than threshold
    - Closing: removes dark regions smaller than threshold
    Works by thresholding at each grey level, filtering connected components by area,
    then reconstructing.
    """
    img = (normalize(image) * 255).astype(int)
    result = np.zeros_like(img)
    
    if mode == 'opening':
        levels = range(1, 256)
    else:  # closing — work on complement
        img = 255 - img
        levels = range(1, 256)
    
    for level in levels:
        binary = (img >= level)
        if not binary.any():
            continue
        cc = connected_components_labeling(binary)
        unique_labels = np.unique(cc[cc > 0])
        for lbl in unique_labels:
            mask = cc == lbl
            area = mask.sum()
            if area >= threshold:
                result[mask] = np.maximum(result[mask], level)
    
    if mode == 'closing':
        result = 255 - result
    
    return result.astype(float) / 255.0

def compute_emap(cube, n_pca=3, area_thresholds=[50, 200, 500]):
    """
    Compute Extended Morphological Attribute Profiles.
    1. PCA reduction to n_pca components
    2. For each component: apply area attribute opening + closing at each threshold
    3. Stack into feature vector per pixel
    """
    H, W, B = cube.shape
    X = cube.reshape(-1, B)
    mu = X.mean(axis=0)
    X_c = X - mu
    cov = (X_c.T @ X_c) / (X_c.shape[0] - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1][:n_pca]
    pca_components = (X_c @ eigvecs[:, idx]).reshape(H, W, n_pca)
    
    profiles = []
    for pc in range(n_pca):
        component = normalize(pca_components[:, :, pc])
        profiles.append(component)  # original component
        for t in area_thresholds:
            print(f'  PC-{pc+1}, area threshold={t}...')
            opened = area_attribute_filter(component, threshold=t, mode='opening')
            closed = area_attribute_filter(component, threshold=t, mode='closing')
            profiles.append(opened)
            profiles.append(closed)
    
    emap = np.stack(profiles, axis=2)  # (H, W, n_features)
    return emap, pca_components

print('Computing EMAPs (PCA + area attribute filters)...')
print('Using reduced thresholds for tractability on 145x145...')
emap, pca_comps = compute_emap(cube, n_pca=3, area_thresholds=[100, 500])
print(f'EMAP feature dimension: {emap.shape[2]} per pixel')

# Nearest-centroid classifier on EMAP features
emap_flat = emap.reshape(N, -1)
classes = np.unique(gt_flat[gt_flat > 0]).astype(int)
centroids = np.zeros((len(classes), emap_flat.shape[1]))
for i, c in enumerate(classes):
    centroids[i] = emap_flat[gt_flat == c].mean(axis=0)

# Classify all pixels
dists = np.zeros((N, len(classes)))
for i, cent in enumerate(centroids):
    dists[:, i] = np.sum((emap_flat - cent)**2, axis=1)
emap_pred = classes[np.argmin(dists, axis=1)]

valid = gt_flat > 0
emap_acc = np.mean(emap_pred[valid] == gt_flat[valid]) * 100
print(f'EMAP + Nearest Centroid accuracy: {emap_acc:.2f}%')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(gt, cmap='nipy_spectral')
axes[0].set_title('Ground Truth'); axes[0].axis('off')
axes[1].imshow(emap_pred.reshape(H, W), cmap='nipy_spectral')
axes[1].set_title(f'EMAP Classification ({emap_acc:.1f}%)'); axes[1].axis('off')
# Show some EMAP feature bands
axes[2].imshow(normalize(emap[:, :, 1]) - normalize(emap[:, :, 0]), cmap='RdBu')
axes[2].set_title('EMAP Differential Profile (PC1)'); axes[2].axis('off')
plt.suptitle('Extended Morphological Attribute Profiles', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/emap_classification.png', dpi=150)
plt.show()

---
## Summary

| Algorithm | Key Technique | From-Scratch Component |
| --- | --- | --- |
| RX Anomaly Detector | Mahalanobis distance via $\Sigma^{-1}$ | Covariance inversion, Tikhonov regularisation |
| VCA Endmember Extraction | Orthogonal subspace projection | SVD projection, iterative null-space selection |
| FCLS Unmixing | Constrained optimisation | Active-set NNLS solver, augmented sum-to-one |
| MNF Transform | Noise-adjusted PCA | Noise covariance estimation, Cholesky whitening |
| Label Propagation | Graph Laplacian diffusion | k-NN graph construction, normalised Laplacian iteration |
| EMAPs | Spectral-spatial attribute profiles | Connected components, area attribute filtering |